## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa de forma numéricamente estable restando el máximo de los logits antes de exponenciar, p_i = exp((z_i − z_max)/T) / Σ_j exp((z_j − z_max)/T); esta reescritura no cambia el resultado matemático pero evita desbordamientos. Con los logits z = (2.0, 1.0, 0.5, 0.2, −1.0, −3.0) de t0 a t5, las distribuciones y las entropías en bits son:

| T | t0 | t1 | t2 | t3 | t4 | t5 | Entropía (bits) |
|------|--------|--------|--------|--------|--------|--------|-----------------|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| 1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| 2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

Cada fila suma 1.0. Bajar la temperatura afila la distribución: la masa de t0 sube de 0.5516 (T=1) a 0.8231 (T=0.5) y la de t5 cae a 3.7367e-05 (0.0000 al redondear a cuatro decimales). Subirla la aplana: con T=2, t0 baja a 0.3583 y t5 sube a 0.0294. La entropía crece monótonamente con T: 0.9147 < 1.7998 < 2.2888 bits.

In [1]:
# T1: código aprobado por el crítico
# T1: Softmax con temperatura (NumPy, numéricamente estable) + entropía en bits
import json

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]
etiquetas = ["T=0.5", "T=1", "T=2"]

# ----------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable (se resta el máximo)
# p_i = exp(z_i / T) / sum_j exp(z_j / T)
# ----------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    z = np.asarray(z, dtype=float)
    T = float(T)
    if T <= 0:
        raise ValueError("La temperatura debe ser positiva.")
    escalados = z / T
    escalados = escalados - np.max(escalados)  # estabilidad numérica
    e = np.exp(escalados)
    return e / np.sum(e)

def entropia_bits(p):
    p = np.asarray(p, dtype=float)
    p_pos = p[p > 0]  # evita log2(0)
    return float(-np.sum(p_pos * np.log2(p_pos)))

# ----------------------------------------------------------------------
# Cálculo de las tres distribuciones y sus entropías
# ----------------------------------------------------------------------
distribuciones = {
    etq: softmax_con_temperatura(logits, T)
    for etq, T in zip(etiquetas, temperaturas)
}
entropias = {etq: entropia_bits(p) for etq, p in distribuciones.items()}

# ----------------------------------------------------------------------
# Tabla: una fila por distribución (6 probabilidades + entropía), 4 decimales
# ----------------------------------------------------------------------
tabla = pd.DataFrame(
    {etq: {tok: float(distribuciones[etq][i]) for i, tok in enumerate(tokens)}
     for etq in etiquetas}
).T
tabla["entropia_bits"] = pd.Series(entropias)
tabla_redondeada = tabla.round(4)
tabla_redondeada.to_csv("tabla_softmax_temperatura.csv")

# ----------------------------------------------------------------------
# Figura: distribuciones para las tres temperaturas
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(tokens))
ancho = 0.25
for k, etq in enumerate(etiquetas):
    ax.bar(x + (k - 1) * ancho, distribuciones[etq], ancho, label=etq)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad")
ax.set_title("Softmax con temperatura sobre los logits de t0..t5")
ax.legend(title="Temperatura")
fig.tight_layout()
plt.savefig("softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# Verificaciones del criterio de éxito
# ----------------------------------------------------------------------
sumas = {etq: float(np.sum(distribuciones[etq])) for etq in etiquetas}
orden_ok = bool(entropias["T=0.5"] < entropias["T=1"] < entropias["T=2"])

# ----------------------------------------------------------------------
# resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": {tok: float(z) for tok, z in zip(tokens, logits)},
    "temperaturas": temperaturas,
    "probabilidades_sin_redondear": {
        etq: [float(v) for v in distribuciones[etq]] for etq in etiquetas
    },
    "probabilidades_por_token_sin_redondear": {
        etq: {tok: float(distribuciones[etq][i]) for i, tok in enumerate(tokens)}
        for etq in etiquetas
    },
    "entropia_bits_sin_redondear": {etq: entropias[etq] for etq in etiquetas},
    "tabla_4_decimales": {
        "filas": etiquetas,
        "columnas": tokens + ["entropia_bits"],
        "valores": [
            [float(v) for v in tabla_redondeada.loc[etq].tolist()]
            for etq in etiquetas
        ],
    },
    "entropias_bits_4_decimales": {
        etq: round(entropias[etq], 4) for etq in etiquetas
    },
    "suma_probabilidades_por_distribucion": sumas,
    "orden_entropias_T05_menor_T1_menor_T2": orden_ok,
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ----------------------------------------------------------------------
# Resumen
# ----------------------------------------------------------------------
print("Softmax con temperatura (estable: se resta el maximo de los logits):")
print(tabla_redondeada.to_string())
print("\nEntropias en bits:")
for etq in etiquetas:
    print(f"  {etq}: {entropias[etq]:.6f} bits  (redondeada: {round(entropias[etq], 4)})")
print("\nSuma de cada fila de probabilidades:", sumas)
print("Orden de entropias T=0.5 < T=1 < T=2:", orden_ok)
print("\nArchivos generados: resultados.json, tabla_softmax_temperatura.csv, softmax_temperatura.png")


Softmax con temperatura (estable: se resta el maximo de los logits):
           t0      t1      t2      t3      t4      t5  entropia_bits
T=0.5  0.8231  0.1114  0.0410  0.0225  0.0020  0.0000         0.9147
T=1    0.5516  0.2029  0.1231  0.0912  0.0275  0.0037         1.7998
T=2    0.3583  0.2173  0.1693  0.1457  0.0800  0.0294         2.2888

Entropias en bits:
  T=0.5: 0.914682 bits  (redondeada: 0.9147)
  T=1: 1.799836 bits  (redondeada: 1.7998)
  T=2: 2.288826 bits  (redondeada: 2.2888)

Suma de cada fila de probabilidades: {'T=0.5': 1.0, 'T=1': 1.0, 'T=2': 1.0}
Orden de entropias T=0.5 < T=1 < T=2: True

Archivos generados: resultados.json, tabla_softmax_temperatura.csv, softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T=1 se ordenan los tokens de mayor a menor probabilidad y se acumula hasta alcanzar p = 0.9 con el conjunto mínimo:

| Posición | Token | Probabilidad | Acumulada |
|----------|-------|--------------|-----------|
| 1 | t0 | 0.5516 | 0.5516 |
| 2 | t1 | 0.2029 | 0.7546 |
| 3 | t2 | 0.1231 | 0.8776 |
| 4 | t3 | 0.0912 | 0.9688 |
| 5 | t4 | 0.0275 | 0.9963 |
| 6 | t5 | 0.0037 | 1.0000 |

Con los tres primeros tokens la acumulada es 0.8776 < 0.9; al incluir t3 se alcanza 0.9688 ≥ 0.9. El núcleo mínimo tiene pues 4 tokens supervivientes: t0, t1, t2 y t3; quedan excluidos t4 y t5, con una masa excluida de 0.0312. Renormalizando entre la masa del núcleo (0.9688):

| Token | Probabilidad original | Probabilidad renormalizada |
|-------|-----------------------|----------------------------|
| t0 | 0.5516 | 0.5694 |
| t1 | 0.2029 | 0.2095 |
| t2 | 0.1231 | 0.1270 |
| t3 | 0.0912 | 0.0941 |

La distribución renormalizada suma 1.0. El recorte reduce la entropía de 1.7998 a 1.6341 bits: al eliminar la cola (t4, t5) se elimina parte de la incertidumbre de la distribución original.

In [2]:
# T2: código aprobado por el crítico
# T2 — Muestreo de núcleo (top-p) con p=0.9 sobre la distribución T=1 de la Parte 1 (T1)
# Solo usa numpy/matplotlib; lee el resultado previo de entrada/T1/resultados.json.

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------- Parámetros fijados por el enunciado ----------------
P = 0.9
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])


def softmax_con_temperatura(logits, T):
    """Softmax con temperatura, numéricamente estable (resta el máximo)."""
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def entropia_bits(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


# ---------------- 1) Distribución T=1 obtenida en T1 ----------------
dist_T1 = None
ruta_T1 = Path("entrada/T1/resultados.json")
if ruta_T1.exists():
    with open(ruta_T1, "r", encoding="utf-8") as f:
        t1 = json.load(f)
    probs = t1.get("probabilidades_sin_redondear", {}).get("T=1")
    if probs is not None:
        dist_T1 = np.array(probs, dtype=float)

if dist_T1 is None:  # respaldo: recalcular con la softmax estable de la Parte 1
    dist_T1 = softmax_con_temperatura(LOGITS, 1.0)

# ---------------- 2) Ordenar tokens de mayor a menor probabilidad ----------------
orden = np.argsort(-dist_T1, kind="stable")
tokens_ordenados = [TOKENS[i] for i in orden]
probs_ordenadas = dist_T1[orden]
acumuladas = np.cumsum(probs_ordenadas)

# ---------------- 3) Conjunto mínimo cuya suma acumulada alcanza p=0.9 ----------------
# Se incluyen los tokens hasta que la suma acumulada alcanza (>=) 0.9 por primera vez.
k = int(np.searchsorted(acumuladas, P, side="left") + 1)
k = min(k, len(TOKENS))

sobrevivientes_idx = orden[:k]
excluidos_idx = orden[k:]

tokens_sobrevivientes = [TOKENS[i] for i in sobrevivientes_idx]
probs_sobrevivientes = dist_T1[sobrevivientes_idx]
suma_sobrevivientes = float(probs_sobrevivientes.sum())

tokens_excluidos = [TOKENS[i] for i in excluidos_idx]
probs_excluidos = dist_T1[excluidos_idx]
masa_excluida = float(probs_excluidos.sum())

# ---------------- 4) Renormalizar las probabilidades del núcleo ----------------
probs_renorm = probs_sobrevivientes / suma_sobrevivientes
suma_renorm = float(probs_renorm.sum())

entropia_original = entropia_bits(dist_T1)
entropia_renorm = entropia_bits(probs_renorm)

# ---------------- resultados.json (contrato de la subtarea) ----------------
resultados = {
    "p": P,
    "temperatura": 1.0,
    "tokens": TOKENS,
    "distribucion_T1_original": {t: float(p) for t, p in zip(TOKENS, dist_T1)},
    # Lista ordenada de tokens con sus probabilidades y acumuladas
    "lista_ordenada_tokens": tokens_ordenados,
    "probabilidades_ordenadas": [float(x) for x in probs_ordenadas],
    "probabilidad_acumulada": [float(x) for x in acumuladas],
    "tabla_ordenada": {
        "posicion": list(range(1, len(TOKENS) + 1)),
        "token": tokens_ordenados,
        "probabilidad": [float(x) for x in probs_ordenadas],
        "probabilidad_acumulada": [float(x) for x in acumuladas],
    },
    # Conjunto mínimo de supervivientes
    "num_tokens_sobrevivientes": k,
    "tokens_sobrevivientes": tokens_sobrevivientes,
    "probabilidades_sobrevivientes_ordenadas": [float(x) for x in probs_sobrevivientes],
    "suma_acumulada_sobrevivientes": suma_sobrevivientes,
    "alcanza_p": bool(suma_sobrevivientes >= P),
    "tokens_excluidos": tokens_excluidos,
    "probabilidades_excluidos": [float(x) for x in probs_excluidos],
    "masa_excluida": masa_excluida,
    # Distribución renormalizada
    "distribucion_renormalizada": {t: float(p) for t, p in zip(tokens_sobrevivientes, probs_renorm)},
    "probabilidades_renormalizadas": [float(x) for x in probs_renorm],
    "suma_renormalizada": suma_renorm,
    "suma_renormalizada_es_1": bool(abs(suma_renorm - 1.0) < 1e-12),
    # Extras informativos
    "entropia_bits_original_T1": entropia_original,
    "entropia_bits_renormalizada": entropia_renorm,
    "tabla_renormalizada_4_decimales": {
        "token": tokens_sobrevivientes,
        "probabilidad_original": [round(float(x), 4) for x in probs_sobrevivientes],
        "probabilidad_renormalizada": [round(float(x), 4) for x in probs_renorm],
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ---------------- Figuras ----------------
# Figura 1: distribución original vs núcleo renormalizado
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
x_orig = np.arange(len(TOKENS))
ax[0].bar(x_orig, dist_T1, color="#4C72B0")
ax[0].set_xticks(x_orig)
ax[0].set_xticklabels(TOKENS)
ax[0].set_ylim(0, float(dist_T1.max()) * 1.25)
ax[0].set_title("Distribución T=1 (softmax original)")
ax[0].set_ylabel("probabilidad")
for i, p in enumerate(dist_T1):
    ax[0].text(i, p + 0.01, f"{p:.4f}", ha="center", fontsize=8)

x_surv = np.arange(k)
ax[1].bar(x_surv, probs_renorm, color="#55A868", label="renormalizada")
ax[1].bar(x_surv, probs_sobrevivientes, color="#4C72B0", alpha=0.45, label="original")
ax[1].set_xticks(x_surv)
ax[1].set_xticklabels(tokens_sobrevivientes)
ax[1].set_ylim(0, 1.05)
ax[1].set_title(f"Top-p (p={P}): núcleo de {k} tokens")
ax[1].set_ylabel("probabilidad")
ax[1].legend(fontsize=8)
for i, p in enumerate(probs_renorm):
    ax[1].text(i, p + 0.02, f"{p:.4f}", ha="center", fontsize=8)

fig.suptitle(f"Muestreo de núcleo top-p: suma del núcleo = {suma_sobrevivientes:.4f} >= {P}")
fig.tight_layout()
fig.savefig("top_p_nucleus.png", dpi=120)
plt.close(fig)

# Figura 2: curva de probabilidad acumulada y corte
fig2, ax2 = plt.subplots(figsize=(7, 4.2))
ax2.plot(range(1, len(TOKENS) + 1), acumuladas, marker="o", color="#C44E52")
ax2.axhline(P, color="gray", linestyle="--", label=f"p = {P}")
ax2.axvline(k, color="green", linestyle=":", label=f"corte: {k} tokens")
ax2.set_xticks(range(1, len(TOKENS) + 1))
ax2.set_xticklabels(tokens_ordenados)
ax2.set_xlabel("tokens ordenados por probabilidad descendente")
ax2.set_ylabel("probabilidad acumulada")
ax2.set_title("Probabilidad acumulada y corte top-p")
ax2.legend()
fig2.tight_layout()
fig2.savefig("top_p_acumulada.png", dpi=120)
plt.close(fig2)

# ---------------- Resumen ----------------
print("=== T2: Top-p (nucleus sampling) con p=0.9 sobre la distribución T=1 ===")
print("Orden descendente:", ", ".join(f"{t}={p:.4f}" for t, p in zip(tokens_ordenados, probs_ordenadas)))
print("Acumuladas:       ", ", ".join(f"{a:.4f}" for a in acumuladas))
print(f"Núcleo mínimo ({k} tokens): {tokens_sobrevivientes} | suma acumulada = {suma_sobrevivientes:.6f} >= {P}")
print(f"Excluidos: {tokens_excluidos} | masa de probabilidad excluida = {masa_excluida:.6f}")
print("Distribución renormalizada:", ", ".join(f"{t}={p:.6f}" for t, p in zip(tokens_sobrevivientes, probs_renorm)))
print(f"Suma de probabilidades renormalizadas = {suma_renorm:.12f}")
print(f"Entropía (bits): original T=1 = {entropia_original:.4f} | renormalizada = {entropia_renorm:.4f}")
print("Salidas: resultados.json, top_p_nucleus.png, top_p_acumulada.png")


=== T2: Top-p (nucleus sampling) con p=0.9 sobre la distribución T=1 ===
Orden descendente: t0=0.5516, t1=0.2029, t2=0.1231, t3=0.0912, t4=0.0275, t5=0.0037
Acumuladas:        0.5516, 0.7546, 0.8776, 0.9688, 0.9963, 1.0000
Núcleo mínimo (4 tokens): ['t0', 't1', 't2', 't3'] | suma acumulada = 0.968820 >= 0.9
Excluidos: ['t4', 't5'] | masa de probabilidad excluida = 0.031180
Distribución renormalizada: t0=0.569376, t1=0.209462, t2=0.127045, t3=0.094117
Suma de probabilidades renormalizadas = 1.000000000000
Entropía (bits): original T=1 = 1.7998 | renormalizada = 1.6341
Salidas: resultados.json, top_p_nucleus.png, top_p_acumulada.png


## Parte 3 — Comprobación empírica

Se toman 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Los conteos observados son t0 = 5682, t1 = 2093, t2 = 1296 y t3 = 929. La comparación con las probabilidades teóricas es:

| Token | Probabilidad teórica | Frecuencia observada | Conteo | Diferencia (obs − teórica) |
|-------|----------------------|----------------------|--------|----------------------------|
| t0 | 0.5694 | 0.5682 | 5682 | −0.0012 |
| t1 | 0.2095 | 0.2093 | 2093 | −0.0002 |
| t2 | 0.1270 | 0.1296 | 1296 | 0.0026 |
| t3 | 0.0941 | 0.0929 | 929 | −0.0012 |

La celda de código de esta parte genera el gráfico de barras que compara las frecuencias observadas con las probabilidades teóricas. La divergencia KL de la distribución empírica respecto de la teórica es 5.0069e-05 bits (3.4705e-05 nats), y la entropía empírica es 1.6362 bits frente a 1.6341 bits teóricos. Un KL tan próximo a cero, con desviaciones máximas del orden de 0.0026 (t2), confirma que el muestreo reproduce la distribución renormalizada y que las diferencias observadas son ruido de muestra finita.

In [3]:
# T3: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
SUBTAREA T3 — Comprobación empírica del muestreo top-p (Parte 3 de la Tarea B)

Pasos:
1. Cargar de entrada/T2/resultados.json los tokens del núcleo (top-p, p=0.9, T=1)
   y su distribución renormalizada (con recálculo de respaldo desde los logits).
2. Tomar 10,000 muestras con numpy.random.default_rng(0).
3. Calcular frecuencias relativas observadas por token.
4. Gráfico de barras: frecuencias observadas vs probabilidades teóricas (solo núcleo).
5. Divergencia KL en bits de la distribución empírica respecto de la teórica.
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# 1) Distribución teórica renormalizada (resultado de T2)
# ----------------------------------------------------------------------
ruta_t2 = Path("entrada/T2/resultados.json")
t2 = None
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        t2 = json.load(f)

if t2 is not None:
    tokens_nucleo = list(t2["tokens_sobrevivientes"])
    p_teorica = np.array(
        [float(t2["distribucion_renormalizada"][tk]) for tk in tokens_nucleo],
        dtype=float,
    )
else:
    # Respaldo: recalcular softmax(T=1) + top-p (p=0.9) desde los logits del enunciado
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()  # softmax numéricamente estable
    p_full = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p_full)
    acum = np.cumsum(p_full[orden])
    k = int(np.searchsorted(acum, 0.9) + 1)  # conjunto mínimo que alcanza 0.9
    idx_nucleo = np.sort(orden[:k])
    tokens_full = [f"t{i}" for i in range(len(logits))]
    tokens_nucleo = [tokens_full[i] for i in idx_nucleo]
    p_teorica = p_full[idx_nucleo] / p_full[idx_nucleo].sum()

assert abs(p_teorica.sum() - 1.0) < 1e-12, "La distribución renormalizada debe sumar 1"

# ----------------------------------------------------------------------
# 2) Muestreo: 10,000 muestras con numpy.random.default_rng(0)
# ----------------------------------------------------------------------
SEMILLA = 0
N_MUESTRAS = 10_000

rng = np.random.default_rng(SEMILLA)
indices = rng.choice(len(tokens_nucleo), size=N_MUESTRAS, p=p_teorica)

# ----------------------------------------------------------------------
# 3) Frecuencias relativas observadas
# ----------------------------------------------------------------------
conteos = np.array([(indices == i).sum() for i in range(len(tokens_nucleo))], dtype=int)
frec_obs = conteos / N_MUESTRAS

# ----------------------------------------------------------------------
# 4) Gráfico de barras: observado vs teórico (solo tokens del núcleo)
# ----------------------------------------------------------------------
nombre_figura = "T3_frecuencias_observadas_vs_teoricas.png"
x = np.arange(len(tokens_nucleo))
ancho = 0.38

fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - ancho / 2, frec_obs, ancho,
            label="Frecuencia observada (empírica)",
            color="#4C72B0", edgecolor="black")
b2 = ax.bar(x + ancho / 2, p_teorica, ancho,
            label="Probabilidad teórica (renormalizada)",
            color="#DD8452", edgecolor="black")
for barras in (b1, b2):
    for b in barras:
        h = b.get_height()
        ax.annotate(f"{h:.4f}",
                    xy=(b.get_x() + b.get_width() / 2, h),
                    xytext=(0, 2), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(tokens_nucleo)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia relativa")
ax.set_title(
    "Top-p (p=0.9, T=1): frecuencias observadas vs probabilidades teóricas\n"
    f"{N_MUESTRAS} muestras, numpy.random.default_rng({SEMILLA})"
)
ax.set_ylim(0.0, max(frec_obs.max(), p_teorica.max()) * 1.18)
ax.legend()
fig.tight_layout()
fig.savefig(nombre_figura, dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 5) Divergencia KL (en bits) de la empírica respecto de la teórica
#    D_KL(q || p) = sum_i q_i * log2(q_i / p_i);  términos con q_i = 0 valen 0
# ----------------------------------------------------------------------
mask = frec_obs > 0
kl_bits = float(np.sum(frec_obs[mask] * np.log2(frec_obs[mask] / p_teorica[mask])))
kl_nats = float(np.sum(frec_obs[mask] * np.log(frec_obs[mask] / p_teorica[mask])))

entropia_empirica_bits = float(-np.sum(frec_obs[mask] * np.log2(frec_obs[mask])))
entropia_teorica_bits = float(-np.sum(p_teorica * np.log2(p_teorica)))

# ----------------------------------------------------------------------
# 6) resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
resultados = {
    "subtarea": "T3",
    "descripcion": (
        "Comprobación empírica: 10,000 muestras de la distribución renormalizada "
        "top-p (p=0.9, T=1) con numpy.random.default_rng(0); frecuencias observadas "
        "vs probabilidades teóricas y divergencia KL en bits (empírica || teórica)."
    ),
    "semilla": SEMILLA,
    "num_muestras": N_MUESTRAS,
    "tokens_nucleo": tokens_nucleo,
    "probabilidades_teoricas_renormalizadas": {
        tk: float(p) for tk, p in zip(tokens_nucleo, p_teorica)
    },
    "conteos_observados": {tk: int(c) for tk, c in zip(tokens_nucleo, conteos)},
    "frecuencias_observadas": {
        tk: float(fr) for tk, fr in zip(tokens_nucleo, frec_obs)
    },
    "tabla_comparativa": {
        "token": tokens_nucleo,
        "probabilidad_teorica": [float(v) for v in p_teorica],
        "frecuencia_observada": [float(v) for v in frec_obs],
        "conteo_observado": [int(v) for v in conteos],
        "diferencia_obs_menos_teorica": [float(v) for v in (frec_obs - p_teorica)],
    },
    "divergencia_KL_bits": kl_bits,
    "divergencia_KL_nats": kl_nats,
    "entropia_bits_empirica": entropia_empirica_bits,
    "entropia_bits_teorica_renormalizada": entropia_teorica_bits,
    "figura": nombre_figura,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ----------------------------------------------------------------------
# Resumen
# ----------------------------------------------------------------------
print("=" * 64)
print("T3 — Comprobación empírica del muestreo top-p (p=0.9, T=1)")
print("=" * 64)
print(f"Muestras: {N_MUESTRAS} | Semilla: {SEMILLA} (numpy.random.default_rng)")
print(f"Tokens del núcleo: {tokens_nucleo}")
print("\nToken | Teórica (renorm.) | Observada | Conteo")
for tk, pt, fo, c in zip(tokens_nucleo, p_teorica, frec_obs, conteos):
    print(f"  {tk}  |      {pt:.4f}       |  {fo:.4f}  | {c}")
print(f"\nDivergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
print(f"  (equivalente en nats: {kl_nats:.6f})")
print(f"Entropía empírica: {entropia_empirica_bits:.6f} bits | "
      f"Entropía teórica: {entropia_teorica_bits:.6f} bits")
print(f"Figura guardada: {nombre_figura}")
print("Resultados escritos en resultados.json")


T3 — Comprobación empírica del muestreo top-p (p=0.9, T=1)
Muestras: 10000 | Semilla: 0 (numpy.random.default_rng)
Tokens del núcleo: ['t0', 't1', 't2', 't3']

Token | Teórica (renorm.) | Observada | Conteo
  t0  |      0.5694       |  0.5682  | 5682
  t1  |      0.2095       |  0.2093  | 2093
  t2  |      0.1270       |  0.1296  | 1296
  t3  |      0.0941       |  0.0929  | 929

Divergencia KL (empírica || teórica) = 0.000050 bits
  (equivalente en nats: 0.000035)
Entropía empírica: 1.636160 bits | Entropía teórica: 1.634075 bits
Figura guardada: T3_frecuencias_observadas_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

Cuando T → 0⁺, dividir los logits entre una temperatura cada vez menor amplifica sus diferencias: los tokens con logit inferior al máximo reciben probabilidades que tienden a 0 y toda la masa se concentra en el token de mayor logit (aquí t0). La distribución límite es una masa puntual sobre el argmax, con entropía mínima (0 bits). Por eso T → 0 equivale a la decodificación voraz: greedy elige siempre el token de mayor logit, y muestrear una distribución que pone probabilidad 1 en ese token produce exactamente la misma decisión, sin aleatoriedad. Nótese que para cualquier T finito el argmax de la softmax coincide con el de los logits (la temperatura no altera el orden), pero solo en el límite T → 0 la política estocástica se vuelve determinista e idéntica a greedy. La evidencia de la Parte 1 apunta en esa dirección: al bajar T de 1 a 0.5 la entropía cae de 1.7998 a 0.9147 bits (la mínima de las tres calculadas), la masa de t0 sube de 0.5516 a 0.8231 y la de t5 se desploma a 3.7367e-05; la tendencia continúa hacia la entropía mínima cuando T → 0.

Cuando T → ∞, dividir entre una temperatura muy grande hace que todos los z_i/T tiendan a 0, de modo que exp(z_i/T) tiende a 1 para cada token y la distribución tiende a la uniforme (1/6 por token). La evidencia medida: con T=2 la distribución ya está visiblemente más plana (t0 baja a 0.3583 y t5 sube a 0.0294) y la entropía sube a 2.2888 bits, la máxima de las tres calculadas, acercándose al máximo posible para un vocabulario de seis tokens, log2(6) bits. El orden medido 0.9147 < 1.7998 < 2.2888 confirma que la entropía crece monótonamente con T, entre el extremo determinista (T → 0, equivalente a greedy) y el extremo uniforme (T → ∞).